In [ ]:
import tables_io, qp
import numpy as np
import os
import matplotlib.pyplot as plt
from nz_data_challenge import submit_utils, metrics, utils, evaluation
from pathlib import Path

In [ ]:
submit_dir = 'submission/rail_knn_4tasks'
model_dir = '../models/rail_knn_4tasks'
public_dir = '../public'
truth_dir = '../reserved'
try:
    os.makedirs(submit_dir)
except:
    pass

In [ ]:
n_bins = 5
z_min = 0
z_max = 1.5
n_grid_points = 151
grid_edges = np.linspace(z_min, z_max, n_grid_points)
grid_centers = 0.5*(grid_edges[0:-1]+grid_edges[1:])
bin_edges = np.array([0., 0.32, 0.47, 0.61, 0.78, 2.5 ])
bin_centers = 0.5*(bin_edges[0:-1]+bin_edges[1:])
bin_sides = np.linspace(-0.5,n_bins-0.5,n_bins+1) 

In [ ]:
taskset = 'taskset_1'
sim = 'cardinal'
scenario = '1yr'
wfd_file = f"{public_dir}/nz_challenge_{taskset}_{sim}_{scenario}_wfd.hdf5"
truth_file = f'{truth_dir}/nz_challenge_{taskset}_{sim}_{scenario}_wfd.hdf5'
nz_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_estimate_wfd.hdf5"
bhat_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_bhat_wfd.hdf5"
nz_estimates = qp.read(nz_file)
test_data = tables_io.read(wfd_file)
truth = tables_io.read(truth_file)
true_redshifts = truth['redshift']
bhat_data = tables_io.read(bhat_file)
bin_assignments = bhat_data['bhat_for_wide_data']
hist_list = []
true_assignments = utils.get_true_bin_assignments(true_redshifts, bin_edges)


In [ ]:
nz_distributions = utils.get_nz_distributions(nz_estimates, grid_centers, 5)
true_distributions = utils.get_true_nz_distributions(true_redshifts, bin_assignments, grid_edges, 5)

In [ ]:
submit_utils.check_files(nz_file, bhat_file, wfd_file, 5)

In [ ]:
assignment_metrics = evaluation.evaluate_bin_assignments(true_assignments, bin_assignments)

In [ ]:
nz_metrics = evaluation.evaluate_distributions(true_distributions, nz_distributions, grid_edges, nz_estimates.ancil['n_object'])

In [ ]:
assignment_metrics

In [ ]:
nz_metrics

In [ ]:
fig_confusion = evaluation.plot_confusion_matrix(true_assignments, bin_assignments, 5)

In [ ]:
fig_nz = evaluation.plot_nz_data(true_distributions, nz_distributions, grid_edges)

In [ ]:
metrics.total_information_loss(true_distributions, nz_distributions, nz_estimates.ancil['n_object'])

In [ ]:
metrics.wasserstein_dist(grid_centers, true_distributions[0], nz_distributions[0])

In [ ]:
taskset = ['taskset_1', 'taskset_2']
sims = ['cardinal', 'flagship']
scenarios = ['1yr', '4yr']
color = ['magenta', 'blue', 'cyan', 'green', 'red']

fig = plt.figure(figsize=(10, 10))
axes = fig.subplots(2, 4)
idx = 0
for taskset in taskset:
    for sim in sims:
        for scenario in scenarios:
            row = idx % 4
            col = int(idx // 4)
            axs = axes[col][row]
            wfd_file = f"{public_dir}/nz_challenge_{taskset}_{sim}_{scenario}_wfd.hdf5"
            truth_file = f'{truth_dir}/nz_challenge_{taskset}_{sim}_{scenario}_wfd.hdf5'
            nz_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_estimate_wfd.hdf5"
            bhat_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_bhat_wfd.hdf5"
            nz_estimates = qp.read(nz_file)
            test_data = tables_io.read(wfd_file)
            truth = tables_io.read(truth_file)
            true_redshifts = truth['redshift']
            bhat_data = tables_io.read(bhat_file)
            bin_assignments = bhat_data['bhat_for_wide_data']
            pdfs = nz_estimates.pdf(grid_centers)
            norms = nz_estimates.ancil['n_object']
            for i in range(n_bins):
                binx = pdfs[i]
                binx_normed = binx/binx.sum()  
                _ = axs.stairs(binx_normed*norms[i], grid_edges, ls='--', color=color[i])
                _ = axs.hist(true_redshifts[bin_assignments==i], grid_edges, histtype='step', color=color[i])
            idx += 1

fig.tight_layout()